<a href="https://colab.research.google.com/github/aayurchik/27_toxicity_prediction/blob/main/%D0%B1%D0%B5%D0%B9%D0%B7%D0%BB%D0%B0%D0%B9%D0%BD_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# PR-AUC основная, ROC-AUC вспомогательная
# Multi-task, три типа представлений

from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

BASE = '/content/drive/MyDrive/Colab Notebooks/'
physchem_df = pd.read_csv(BASE + '1_targets_and_features.csv')
targets_df = pd.read_csv(BASE + '2_targets_only.csv')
# физхим признаки
feat_cols = [c for c in physchem_df.columns if c.startswith('f')]
X_phys = physchem_df[feat_cols].values
smiles = physchem_df['smiles'].values
# таргеты
targets_df = targets_df.set_index('smiles')
target_cols = [c for c in targets_df.columns if c != 'smiles']
# Morgan fingerprints
fp_sparse = sparse.load_npz(BASE + 'morgan_fp.npz')
fp_smiles = pd.read_csv(BASE + 'fp_smiles.csv')
fp_map = dict(zip(fp_smiles['smiles'], fp_smiles['index']))
# Mol2Vec
mol2vec_df = pd.read_csv(BASE + 'mol2vec_embeddings.csv')
mol2vec_df = mol2vec_df.set_index('smiles')
m2v_cols = [c for c in mol2vec_df.columns]
# Общий датасет (пересечение)
valid_smiles = [
    s for s in smiles
    if (s in fp_map) and (s in mol2vec_df.index) and (s in targets_df.index)]
print(f"Общих молекул: {len(valid_smiles)}") # один и тот же набор SMILES, для которого есть все типы признаков

# индексы
idx_map = {s: i for i, s in enumerate(smiles)}
phys_idx = [idx_map[s] for s in valid_smiles]
fp_idx = [fp_map[s] for s in valid_smiles]
X_phys = X_phys[phys_idx]
X_fp = fp_sparse[fp_idx].toarray()
X_m2v = mol2vec_df.loc[valid_smiles, m2v_cols].values
Y = targets_df.loc[valid_smiles, target_cols].values.astype(float)

Mounted at /content/drive
Общих молекул: 10000


In [2]:
# Split (простой пока, без scaffold)
idx = np.arange(len(valid_smiles))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42)

# Метрики
def pr_auc(y_true, y_prob):
    mask = ~np.isnan(y_true)
    if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
        return np.nan
    return average_precision_score(y_true[mask], y_prob[mask])

def roc_auc(y_true, y_prob):
    mask = ~np.isnan(y_true)
    if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
        return np.nan
    return roc_auc_score(y_true[mask], y_prob[mask])

# Обучение по таргетам
def train_and_eval(X_tr, X_te, Y_tr, Y_te, model, model_name, feat_name):
    rows = []
    t0 = time.time()
    for i, cat in enumerate(target_cols):
        y_tr = Y_tr[:, i]
        y_te = Y_te[:, i]
        train_mask = ~np.isnan(y_tr)
        test_mask  = ~np.isnan(y_te)
        if train_mask.sum() < 20:
            continue
        if len(np.unique(y_tr[train_mask])) < 2:
            continue
        m = model()
        m.fit(X_tr[train_mask], y_tr[train_mask])
        prob = m.predict_proba(X_te[test_mask])[:, 1]
        pr  = pr_auc(y_te[test_mask], prob)
        roc = roc_auc(y_te[test_mask], prob)
        rows.append({
            'features': feat_name,
            'model': model_name,
            'category': cat,
            'PR-AUC': pr,
            'ROC-AUC': roc})
    train_time = time.time() - t0
    # macro mean
    pr_vals  = [r['PR-AUC'] for r in rows if not np.isnan(r['PR-AUC'])]
    roc_vals = [r['ROC-AUC'] for r in rows if not np.isnan(r['ROC-AUC'])]
    rows.append({
        'features': feat_name,
        'model': model_name,
        'category': '>>> MACRO MEAN',
        'PR-AUC': np.mean(pr_vals),
        'ROC-AUC': np.mean(roc_vals),
        'train_time_s': round(train_time, 2)})
    print(f"[{feat_name:10s} | {model_name:6s}] "
          f"PR-AUC={np.mean(pr_vals):.4f}  "
          f"time={train_time:.1f}s")
    return rows

# Модели
def logreg():
    return LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='saga',
        n_jobs=-1)
def knn():
    return KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1)

In [3]:
# Запуск baseline
all_rows = []
representations = [
    ('PhysChem', X_phys),
    ('MorganFP', X_fp),
    ('Mol2Vec',  X_m2v)]
for feat_name, X in representations:
    X_tr = X[train_idx]
    X_te = X[test_idx]
    Y_tr = Y[train_idx]
    Y_te = Y[test_idx]
    all_rows += train_and_eval(X_tr, X_te, Y_tr, Y_te,
                               logreg, 'LogReg', feat_name)
    all_rows += train_and_eval(X_tr, X_te, Y_tr, Y_te,
                               knn, 'KNN', feat_name)
results_df = pd.DataFrame(all_rows)

# Сводка
summary = results_df[results_df['category'] == '>>> MACRO MEAN'].sort_values('PR-AUC', ascending=False)
print("SUMMARY")
print(summary[['features', 'model', 'PR-AUC', 'ROC-AUC', 'train_time_s']])

[PhysChem   | LogReg] PR-AUC=0.5784  time=17.4s
[PhysChem   | KNN   ] PR-AUC=0.5348  time=0.3s
[MorganFP   | LogReg] PR-AUC=0.5975  time=362.1s
[MorganFP   | KNN   ] PR-AUC=0.5072  time=2.7s
[Mol2Vec    | LogReg] PR-AUC=0.5801  time=5.1s
[Mol2Vec    | KNN   ] PR-AUC=0.5497  time=0.7s
SUMMARY
    features   model    PR-AUC   ROC-AUC  train_time_s
41  MorganFP  LogReg  0.597544  0.654942        362.06
69   Mol2Vec  LogReg  0.580143  0.643142          5.08
13  PhysChem  LogReg  0.578400  0.646239         17.44
83   Mol2Vec     KNN  0.549679  0.637025          0.74
27  PhysChem     KNN  0.534799  0.599097          0.26
55  MorganFP     KNN  0.507202  0.592439          2.69


In [5]:
# комбинации признаков
X_concat_light = np.concatenate([X_phys, X_m2v], axis=1)
X_concat_all   = np.concatenate([X_phys, X_fp, X_m2v], axis=1)
for feat_name, X in [
    ('PhysChem+Mol2Vec', X_concat_light),
    ('ALL', X_concat_all)]:
    X_tr = X[train_idx]
    X_te = X[test_idx]
    Y_tr = Y[train_idx]
    Y_te = Y[test_idx]
    all_rows += train_and_eval(
        X_tr, X_te, Y_tr, Y_te,
        logreg, 'LogReg', feat_name)
    all_rows += train_and_eval(
        X_tr, X_te, Y_tr, Y_te,
        knn, 'KNN', feat_name)
results_df = pd.DataFrame(all_rows)
summary = results_df[results_df['category'] == '>>> MACRO MEAN'].copy()
summary = summary[['features', 'model', 'PR-AUC', 'ROC-AUC', 'train_time_s']]
summary = summary.sort_values('PR-AUC', ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

[PhysChem+Mol2Vec | LogReg] PR-AUC=0.5812  time=69.8s
[PhysChem+Mol2Vec | KNN   ] PR-AUC=0.5374  time=0.5s
[ALL        | LogReg] PR-AUC=0.5960  time=427.9s
[ALL        | KNN   ] PR-AUC=0.5381  time=2.7s
        features  model   PR-AUC  ROC-AUC  train_time_s
        MorganFP LogReg 0.597544 0.654942        362.06
             ALL LogReg 0.595968 0.670547        427.94
PhysChem+Mol2Vec LogReg 0.581161 0.653857         69.83
         Mol2Vec LogReg 0.580143 0.643142          5.08
        PhysChem LogReg 0.578400 0.646239         17.44
         Mol2Vec    KNN 0.549679 0.637025          0.74
             ALL    KNN 0.538112 0.603445          2.71
PhysChem+Mol2Vec    KNN 0.537413 0.605132          0.50
        PhysChem    KNN 0.534799 0.599097          0.26
        MorganFP    KNN 0.507202 0.592439          2.69


Наилучшие результаты показывают Morgan fingerprints. Линейная модель стабильно превосходит KNN, особенно в условиях высокой размерности признаков.Комбинирование признаков не приводит к улучшению качества, это указывает на
избыточность, неспособность линейной модели эффективно интегрировать разнородные представления. Mol2Vec и PhysChem дают сопоставимые, но более слабые результаты, что говорит о потере части структурной информации.
фингерпринты обеспечивают лучший баланс качества и интерпретируемости, несмотря на высокую вычислительную стоимость.

In [9]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
# scaffold
def get_scaffold(sm):
    try:
        mol = Chem.MolFromSmiles(sm)
        if mol is None:
            return None
        Chem.SanitizeMol(mol)
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None:
            return None
        return Chem.MolToSmiles(scaf)
    except Exception:
        return None

scaffolds = [get_scaffold(s) for s in valid_smiles]
valid_idx = [i for i, sc in enumerate(scaffolds) if sc is not None]
scaffolds = [scaffolds[i] for i in valid_idx]

X_phys_scaf = X_phys[valid_idx]
X_fp_scaf = X_fp[valid_idx]
X_m2v_scaf = X_m2v[valid_idx]
Y_scaf = Y[valid_idx]
X_light_scaf = np.concatenate([X_phys_scaf, X_m2v_scaf], axis=1)
X_all_scaf = np.concatenate([X_phys_scaf, X_fp_scaf, X_m2v_scaf], axis=1)
scaffold_dict = {}
for i, scaf in enumerate(scaffolds):
    scaffold_dict.setdefault(scaf, []).append(i)
scaffold_sets = sorted(scaffold_dict.values(), key=lambda x: -len(x))
train_idx_scaf, test_idx_scaf = [], []
test_size = int(0.2 * len(scaffolds))

for group in scaffold_sets:
    if len(test_idx_scaf) + len(group) <= test_size:
        test_idx_scaf.extend(group)
    else:
        train_idx_scaf.extend(group)

train_idx_scaf = np.array(train_idx_scaf)
test_idx_scaf  = np.array(test_idx_scaf)

print(f"Train: {len(train_idx_scaf)}, Test: {len(test_idx_scaf)}")

# запускаем только топовые варианты
representations = [
    ('MorganFP_scaf', X_fp_scaf),
    ('ALL_scaf', X_all_scaf),
    ('PhysChem+Mol2Vec_scaf', X_light_scaf),]

scaffold_rows = []
for feat_name, X in representations:
    X_tr = X[train_idx_scaf]
    X_te = X[test_idx_scaf]
    Y_tr = Y_scaf[train_idx_scaf]
    Y_te = Y_scaf[test_idx_scaf]

    scaffold_rows += train_and_eval(
        X_tr, X_te, Y_tr, Y_te,
        logreg, 'LogReg', feat_name)
    scaffold_rows += train_and_eval(
        X_tr, X_te, Y_tr, Y_te,
        knn, 'KNN', feat_name)
scaffold_df = pd.DataFrame(scaffold_rows)
# summary
scaf_summary = scaffold_df[scaffold_df['category'] == '>>> MACRO MEAN'].sort_values('PR-AUC', ascending=False)
print(scaf_summary[['features', 'model', 'PR-AUC', 'ROC-AUC']])

Train: 8000, Test: 2000
[MorganFP_scaf | LogReg] PR-AUC=0.5484  time=361.2s
[MorganFP_scaf | KNN   ] PR-AUC=0.5012  time=2.6s
[ALL_scaf   | LogReg] PR-AUC=0.5713  time=443.1s
[ALL_scaf   | KNN   ] PR-AUC=0.5208  time=2.5s
[PhysChem+Mol2Vec_scaf | LogReg] PR-AUC=0.5676  time=65.6s
[PhysChem+Mol2Vec_scaf | KNN   ] PR-AUC=0.5135  time=0.5s
                 features   model    PR-AUC   ROC-AUC
38               ALL_scaf  LogReg  0.571323  0.609307
64  PhysChem+Mol2Vec_scaf  LogReg  0.567592  0.609966
12          MorganFP_scaf  LogReg  0.548430  0.578900
51               ALL_scaf     KNN  0.520822  0.580455
77  PhysChem+Mol2Vec_scaf     KNN  0.513491  0.578621
25          MorganFP_scaf     KNN  0.501150  0.540330


при переходе от случайного разбиения к scaffold-based split наблюдается снижение качества моделей, что подтверждает наличие структурного domain shift в химическом пространстве. fingerprints показывают наилучший результат при случайном разбиении, при scaffold split уступают комбинированным представлениям, что указывает на ограниченную способность FP к обобщению на новые химические скелеты. Наиболее устойчивым оказывается комбинированное представление PhysChem + Mol2Vec + FP. Logistic Regression стабильно превосходит KNN во всех сценариях, особенно в высокоразмерных пространствах.



In [11]:
np.savez_compressed(BASE + 'aligned_data.npz',
    X_phys=X_phys,
    X_fp=X_fp,
    X_m2v=X_m2v,
    Y=Y,
    smiles=valid_smiles)
np.save(BASE + 'train_idx.npy', train_idx)
np.save(BASE + 'test_idx.npy', test_idx)
np.save(BASE + 'target_cols.npy', target_cols)
print("Saved: aligned_data.npz, train/test indices")

Saved: aligned_data.npz, train/test indices


In [12]:
import os
print(os.path.exists('/content/drive/MyDrive/Colab Notebooks/aligned_data.npz'))

True


In [7]:
# !pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 16.3 MB/s eta 0:00:00
